In [1]:
#Load the cleaned and merged datasetimport pandas as pd

import pandas as pd
import numpy as np

def to_naive_utc(series):
    """Parse a column as datetime and strip timezone info (if any), so it can
    be merged against timezone-naive datetime columns from other sources."""
    s = pd.to_datetime(series)
    if s.dt.tz is not None:
        s = s.dt.tz_convert("UTC").dt.tz_localize(None)
    return s

df = pd.read_csv("../data/processed/kathmandu_merged.csv")
df["datetime_utc"] = to_naive_utc(df["datetime_utc"])
df = df.sort_values(["station", "datetime_utc"]).reset_index(drop=True)

print(f"pm1/pm25 correlation before drop: {df['pm1'].corr(df['pm25']):.4f}")
df = df.drop(columns=["pm1"])

print(df.shape)
df.head()

pm1/pm25 correlation before drop: 0.9939
(13575, 7)


,datetime_utc,pm25,relativehumidity,temperature,um003,was_missing,station
0,2025-11-25 08:15:00,38.9,38.5,21.3,1370.0,False,mid_baneshwor
1,2025-11-25 09:15:00,44.9,38.1,21.9,1790.0,False,mid_baneshwor
2,2025-11-25 10:15:00,48.9,39.3,21.3,1980.0,False,mid_baneshwor
3,2025-11-25 11:15:00,48.0,41.2,20.2,1980.0,False,mid_baneshwor
4,2025-11-25 12:15:00,49.4,40.9,20.1,2060.0,False,mid_baneshwor


In [2]:
#Time based features

df["hour"] = df["datetime_utc"].dt.hour
df["day_of_week"] = df["datetime_utc"].dt.dayofweek   # Monday=0 ... Sunday=6
df["month"] = df["datetime_utc"].dt.month

# Nepal's weekend is Saturday; adjust if needed
df["is_weekend"] = df["day_of_week"].isin([5]).astype(int)  # 5 = Saturday and True = 1 and False = 0

df[["datetime_utc", "hour", "day_of_week", "month", "is_weekend"]].head()


,datetime_utc,hour,day_of_week,month,is_weekend
0,2025-11-25 08:15:00,8,1,11,0
1,2025-11-25 09:15:00,9,1,11,0
2,2025-11-25 10:15:00,10,1,11,0
3,2025-11-25 11:15:00,11,1,11,0
4,2025-11-25 12:15:00,12,1,11,0


In [ ]:
# Cyclical encoding for hour and month 
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

df[["hour", "hour_sin", "hour_cos", "month", "month_sin", "month_cos"]].head()

,hour,hour_sin,hour_cos,month,month_sin,month_cos
0,8,8.660254e-01,-0.500000,11,-0.5,0.866025
1,9,7.071068e-01,-0.707107,11,-0.5,0.866025
2,10,5.000000e-01,-0.866025,11,-0.5,0.866025
3,11,2.588190e-01,-0.965926,11,-0.5,0.866025
4,12,1.224647e-16,-1.000000,11,-0.5,0.866025


In [4]:
lag_hours = [1, 2, 6, 7, 8, 9, 24, 48]   

for lag in lag_hours:
    df[f"pm25_lag_{lag}h"] = df.groupby("station")["pm25"].shift(lag)

df[["station", "datetime_utc", "pm25"] + [f"pm25_lag_{lag}h" for lag in lag_hours]].head(10)

,station,datetime_utc,pm25,pm25_lag_1h,pm25_lag_2h,pm25_lag_6h,pm25_lag_7h,pm25_lag_8h,pm25_lag_9h,pm25_lag_24h,pm25_lag_48h
0,mid_baneshwor,2025-11-25 08:15:00,38.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mid_baneshwor,2025-11-25 09:15:00,44.9,38.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,mid_baneshwor,2025-11-25 10:15:00,48.9,44.9,38.9,NaN,NaN,NaN,NaN,NaN,NaN
3,mid_baneshwor,2025-11-25 11:15:00,48.0,48.9,44.9,NaN,NaN,NaN,NaN,NaN,NaN
4,mid_baneshwor,2025-11-25 12:15:00,49.4,48.0,48.9,NaN,NaN,NaN,NaN,NaN,NaN
5,mid_baneshwor,2025-11-25 13:15:00,58.5,49.4,48.0,NaN,NaN,NaN,NaN,NaN,NaN
6,mid_baneshwor,2025-11-25 14:15:00,110.0,58.5,49.4,38.9,NaN,NaN,NaN,NaN,NaN
7,mid_baneshwor,2025-11-25 15:15:00,119.0,110.0,58.5,44.9,38.9,NaN,NaN,NaN,NaN
8,mid_baneshwor,2025-11-25 16:15:00,102.0,119.0,110.0,48.9,44.9,38.9,NaN,NaN,NaN
9,mid_baneshwor,2025-11-25 17:15:00,65.0,102.0,119.0,48.0,48.9,44.9,38.9,NaN,NaN


In [5]:
rolling_windows = [6, 24]

for window in rolling_windows:
    df[f"pm25_rolling_mean_{window}h"] = (
        df.groupby("station")["pm25"]
        .transform(lambda x: x.rolling(window=window, min_periods=1).mean())
    )
    df[f"pm25_rolling_std_{window}h"] = (
        df.groupby("station")["pm25"]
        .transform(lambda x: x.rolling(window=window, min_periods=1).std())
    )

df[["station", "datetime_utc", "pm25", "pm25_rolling_mean_6h", "pm25_rolling_std_6h",
    "pm25_rolling_mean_24h", "pm25_rolling_std_24h"]].head(10)

,station,datetime_utc,pm25,pm25_rolling_mean_6h,pm25_rolling_std_6h,pm25_rolling_mean_24h,pm25_rolling_std_24h
0,mid_baneshwor,2025-11-25 08:15:00,38.9,38.900000,NaN,38.900000,NaN
1,mid_baneshwor,2025-11-25 09:15:00,44.9,41.900000,4.242641,41.900000,4.242641
2,mid_baneshwor,2025-11-25 10:15:00,48.9,44.233333,5.033223,44.233333,5.033223
3,mid_baneshwor,2025-11-25 11:15:00,48.0,45.175000,4.520601,45.175000,4.520601
4,mid_baneshwor,2025-11-25 12:15:00,49.4,46.020000,4.347068,46.020000,4.347068
5,mid_baneshwor,2025-11-25 13:15:00,58.5,48.100000,6.409056,48.100000,6.409056
6,mid_baneshwor,2025-11-25 14:15:00,110.0,59.950000,24.939186,56.942857,24.116444
7,mid_baneshwor,2025-11-25 15:15:00,119.0,72.300000,33.030531,64.700000,31.303400
8,mid_baneshwor,2025-11-25 16:15:00,102.0,81.150000,32.618016,68.844444,31.811991
9,mid_baneshwor,2025-11-25 17:15:00,65.0,83.983333,29.777200,68.460000,30.017262


In [6]:
#Set target variable: PM2.5 concentration in the next hour (1-hour ahead prediction)
df["target_pm25_next_1h"] = df.groupby("station")["pm25"].shift(-1)

df[["station", "datetime_utc", "pm25", "target_pm25_next_1h"]].head(10)

,station,datetime_utc,pm25,target_pm25_next_1h
0,mid_baneshwor,2025-11-25 08:15:00,38.9,44.9
1,mid_baneshwor,2025-11-25 09:15:00,44.9,48.9
2,mid_baneshwor,2025-11-25 10:15:00,48.9,48.0
3,mid_baneshwor,2025-11-25 11:15:00,48.0,49.4
4,mid_baneshwor,2025-11-25 12:15:00,49.4,58.5
5,mid_baneshwor,2025-11-25 13:15:00,58.5,110.0
6,mid_baneshwor,2025-11-25 14:15:00,110.0,119.0
7,mid_baneshwor,2025-11-25 15:15:00,119.0,102.0
8,mid_baneshwor,2025-11-25 16:15:00,102.0,65.0
9,mid_baneshwor,2025-11-25 17:15:00,65.0,71.6


In [7]:
teku_weather = pd.read_csv("../data/raw/teku_weather.csv")
mid_b_weather = pd.read_csv("../data/raw/mid_baneshwor_weather.csv")

teku_weather = teku_weather.replace(-999.0, np.nan)
mid_b_weather = mid_b_weather.replace(-999.0, np.nan)

teku_weather["datetime_utc"] = to_naive_utc(teku_weather["datetime_utc"])
mid_b_weather["datetime_utc"] = to_naive_utc(mid_b_weather["datetime_utc"])

weather_columns = ["datetime_utc", "WS10M", "WD10M", "PRECTOTCORR"]
teku_weather = teku_weather[weather_columns]
mid_b_weather = mid_b_weather[weather_columns]

print(teku_weather.isna().sum())
print(mid_b_weather.isna().sum())

datetime_utc     0
WS10M           96
WD10M           96
PRECTOTCORR     96
dtype: int64
datetime_utc     0
WS10M           96
WD10M           96
PRECTOTCORR     96
dtype: int64


In [8]:
teku_weather["station"] = "teku"
mid_b_weather["station"] = "mid_baneshwor"

weather_combined = pd.concat([teku_weather, mid_b_weather], ignore_index=True)

print(weather_combined.shape)
print(weather_combined.dtypes)
weather_combined.head()

(13632, 5)
datetime_utc    datetime64[ns]
WS10M                  float64
WD10M                  float64
PRECTOTCORR            float64
station                 object
dtype: object


,datetime_utc,WS10M,WD10M,PRECTOTCORR,station
0,2025-11-25 00:00:00,1.30,18.9,0.0,teku
1,2025-11-25 01:00:00,1.18,20.9,0.0,teku
2,2025-11-25 02:00:00,0.64,12.7,0.0,teku
3,2025-11-25 03:00:00,1.21,237.5,0.0,teku
4,2025-11-25 04:00:00,2.20,228.5,0.0,teku


In [9]:
df["datetime_hour"] = df["datetime_utc"].dt.floor("h")
weather_combined["datetime_hour"] = weather_combined["datetime_utc"].dt.floor("h")

print(df[["datetime_utc", "datetime_hour"]].head())
print(weather_combined[["datetime_utc", "datetime_hour"]].head())

         datetime_utc       datetime_hour
0 2025-11-25 08:15:00 2025-11-25 08:00:00
1 2025-11-25 09:15:00 2025-11-25 09:00:00
2 2025-11-25 10:15:00 2025-11-25 10:00:00
3 2025-11-25 11:15:00 2025-11-25 11:00:00
4 2025-11-25 12:15:00 2025-11-25 12:00:00
         datetime_utc       datetime_hour
0 2025-11-25 00:00:00 2025-11-25 00:00:00
1 2025-11-25 01:00:00 2025-11-25 01:00:00
2 2025-11-25 02:00:00 2025-11-25 02:00:00
3 2025-11-25 03:00:00 2025-11-25 03:00:00
4 2025-11-25 04:00:00 2025-11-25 04:00:00


In [10]:
rows_before = len(df)

df = df.merge(
    weather_combined.drop(columns=["datetime_utc"]),
    on=["datetime_hour", "station"],
    how="left"
)
df = df.drop(columns=["datetime_hour"])

rows_after = len(df)
print(f"Rows before merge: {rows_before}, after merge: {rows_after}")
assert rows_before == rows_after, "Row count changed after merge - check for duplicate timestamps."

print(df[["WS10M", "WD10M", "PRECTOTCORR"]].isna().sum())

Rows before merge: 13575, after merge: 13575
WS10M          143
WD10M          143
PRECTOTCORR    143
dtype: int64


In [ ]:
# Cyclical encoding for wind direction 
df["wind_dir_sin"] = np.sin(2 * np.pi * df["WD10M"] / 360)
df["wind_dir_cos"] = np.cos(2 * np.pi * df["WD10M"] / 360)

df[["WD10M", "wind_dir_sin", "wind_dir_cos"]].head()

,WD10M,wind_dir_sin,wind_dir_cos
0,239.1,-0.858065,-0.513541
1,243.3,-0.893371,-0.449319
2,243.1,-0.891798,-0.452435
3,244.1,-0.899558,-0.436802
4,258.5,-0.979925,-0.199368


In [12]:
#Station as categorical variable (one-hot encoding)
df = pd.get_dummies(df, columns=["station"], prefix="station")

df.filter(like="station_").head()

,station_mid_baneshwor,station_teku
0,True,False
1,True,False
2,True,False
3,True,False
4,True,False


In [13]:
#Drop rows with missing target values (the last row for each station will have a NaN target since there's no next hour data)

before = len(df)
df = df.dropna(subset=["target_pm25_next_1h"]).reset_index(drop=True)
after = len(df)

print(f"Dropped {before - after} rows with missing target ({(before-after)/before:.1%})")
print(f"Remaining rows: {after}")

Dropped 5232 rows with missing target (38.5%)
Remaining rows: 8343


In [14]:
print(df.shape)
print(df.isna().sum())
df.head()

(8343, 34)
datetime_utc               0
pm25                      31
relativehumidity          31
temperature               31
um003                     31
was_missing                0
hour                       0
day_of_week                0
month                      0
is_weekend                 0
hour_sin                   0
hour_cos                   0
month_sin                  0
month_cos                  0
pm25_lag_1h               63
pm25_lag_2h               95
pm25_lag_6h              202
pm25_lag_7h              226
pm25_lag_8h              249
pm25_lag_9h              272
pm25_lag_24h             551
pm25_lag_48h             870
pm25_rolling_mean_6h      29
pm25_rolling_std_6h       60
pm25_rolling_mean_24h     20
pm25_rolling_std_24h      42
target_pm25_next_1h        0
WS10M                    141
WD10M                    141
PRECTOTCORR              141
wind_dir_sin             141
wind_dir_cos             141
station_mid_baneshwor      0
station_teku               0
dty

,datetime_utc,pm25,relativehumidity,temperature,um003,was_missing,hour,day_of_week,month,is_weekend,...,pm25_rolling_mean_24h,pm25_rolling_std_24h,target_pm25_next_1h,WS10M,WD10M,PRECTOTCORR,wind_dir_sin,wind_dir_cos,station_mid_baneshwor,station_teku
0,2025-11-25 08:15:00,38.9,38.5,21.3,1370.0,False,8,1,11,0,...,38.900000,NaN,44.9,3.66,239.1,0.0,-0.858065,-0.513541,True,False
1,2025-11-25 09:15:00,44.9,38.1,21.9,1790.0,False,9,1,11,0,...,41.900000,4.242641,48.9,3.34,243.3,0.0,-0.893371,-0.449319,True,False
2,2025-11-25 10:15:00,48.9,39.3,21.3,1980.0,False,10,1,11,0,...,44.233333,5.033223,48.0,2.41,243.1,0.0,-0.891798,-0.452435,True,False
3,2025-11-25 11:15:00,48.0,41.2,20.2,1980.0,False,11,1,11,0,...,45.175000,4.520601,49.4,1.58,244.1,0.0,-0.899558,-0.436802,True,False
4,2025-11-25 12:15:00,49.4,40.9,20.1,2060.0,False,12,1,11,0,...,46.020000,4.347068,58.5,1.05,258.5,0.0,-0.979925,-0.199368,True,False


In [15]:
import os
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/kathmandu_features.csv", index=False)
print(f"Saved {len(df)} rows to ../data/processed/kathmandu_features.csv")

Saved 8343 rows to ../data/processed/kathmandu_features.csv
